# Clip Extraction — YOLOv8 + HRNet + BoT-SORT

### Alur kerja:
1. Semua video di `dataset/data_in/` diproses otomatis (batch)
2. Tiap video menghasilkan: `.csv` + `.pkl` + video output anotasi
3. Buka video output → isi label di `.csv` → jalankan cell labeling
4. Combine semua `.pkl` berlabel → siap training PoseC3D


## 1. Import & Konfigurasi

In [2]:
# Copyright (c) CIIS-Lab. All rights reserved.
import os
import os.path as osp
import copy as cp
import tempfile
import gc
import glob


import cv2
import mmcv
import mmengine
import numpy as np
import torch

from mmaction.apis import pose_inference
from mmaction.registry import VISUALIZERS
from mmaction.utils import frame_extract
from ultralytics import YOLO

# BoT-SORT dari library boxmot
from boxmot import BoTSORT
from pathlib import Path

import moviepy.editor as mpy

In [3]:
FONTFACE  = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 1
THICKNESS = 2
LINETYPE  = 1

In [25]:
# ============================================================
#  KONFIGURASI
# ============================================================

# Folder input & output
INPUT_DIR  = '03_dataset/data_baru/knockout'        # folder berisi semua video .mp4
OUTPUT_DIR = '03_dataset/data_out/1_out'                        # folder output video anotasi
PKL_DIR    = '03_dataset/data_in/1_csv'                # folder simpan .csv dan .pkl mentah per video

# Human detection
model_path    = os.path.abspath('01_asset_tools/models/yolov8m.pt')
detector      = YOLO(model_path)
det_score_thr = 0.3

# Pose estimation (HRNet)
pose_config     = 'mmaction2/demo/demo_configs/td-hm_hrnet-w32_8xb64-210e_coco-256x192_infer.py'
pose_checkpoint = 'https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth'

# Label map
label_map_stdet = '03_dataset/ciis_label_map.txt'

# Clip & output
predict_stepsize = 4
output_fps       = 12
device           = 'cuda:0'

# BoT-SORT
BOTSORT_REID_WEIGHTS = Path('01_asset_tools/models/osnet_x0_25_msmt17.pt')
BOTSORT_DEVICE       = device
BOTSORT_HALF         = True   # RTX 3090 support FP16


## 2. Helper Functions

In [6]:
def hex2color(h):
    return (int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16))

PLATEBLUE = [hex2color(h) for h in '03045e-023e8a-0077b6-0096c7-00b4d8-48cae4'.split('-')]

def abbrev(name):
    while name.find('(') != -1:
        st, ed = name.find('('), name.find(')')
        name = name[:st] + '...' + name[ed + 1:]
    return name


def _cal_iou(box1, box2):
    """Hitung Intersection over Union antara dua bbox [x1,y1,x2,y2]."""
    xmin1, ymin1, xmax1, ymax1 = box1
    xmin2, ymin2, xmax2, ymax2 = box2
    s1    = max(0, xmax1 - xmin1) * max(0, ymax1 - ymin1)
    s2    = max(0, xmax2 - xmin2) * max(0, ymax2 - ymin2)
    xi    = max(0, min(xmax1, xmax2) - max(xmin1, xmin2))
    yi    = max(0, min(ymax1, ymax2) - max(ymin1, ymin2))
    inter = xi * yi
    union = s1 + s2 - inter
    return inter / union if union > 0 else 0


def expand_bbox(bbox, h, w, ratio=1.25):
    x1, y1, x2, y2 = bbox
    cx, cy   = (x1 + x2) // 2, (y1 + y2) // 2
    sq       = max(x2 - x1, y2 - y1) * ratio
    return (max(0, int(cx - sq/2)), max(0, int(cy - sq/2)),
            min(int(cx + sq/2), w),  min(int(cy + sq/2), h))


def pack_result(human_detection, result, img_h, img_w):
    human_detection[:, 0::2] /= img_w
    human_detection[:, 1::2] /= img_h
    if result is None:
        return None
    results = []
    for prop, res in zip(human_detection, result):
        res.sort(key=lambda x: -x[1])
        results.append((prop.data.cpu().numpy(), [x[0] for x in res], [x[1] for x in res]))
    return results

## 3. YOLOv8 + BoT-SORT Tracking

**Output utama:** `tracked_detections[frame_idx]` = list dict dengan key:
- `track_id` — ID orang yang **konsisten** sepanjang video
- `bbox` — [x1, y1, x2, y2]
- `score` — confidence

Ini menggantikan `human_detections` lama yang tidak punya track_id.

In [7]:
def run_detection_and_tracking(frame_paths, detector, det_score_thr,
                                reid_weights, track_device, half=True):
    """
    YOLOv8 + BoT-SORT untuk setiap frame.

    Returns:
        tracked_detections : list[list[dict]]  — per frame, per orang: {track_id, bbox, score}
        human_detections   : list[np.ndarray]  — per frame, array [N,5] untuk pose_inference
    """
    # Inisialisasi BoT-SORT
    # Menggabungkan Kalman Filter (prediksi posisi) + Re-ID (pengenalan penampilan)
    tracker = BoTSORT(
        model_weights=reid_weights,
        device=track_device,
        fp16=half,
        track_high_thresh=det_score_thr,  # threshold deteksi utama
        track_low_thresh=0.1,             # threshold deteksi lemah (second association)
        new_track_thresh=det_score_thr,   # threshold buat track baru
        track_buffer=50,                  # frame sebelum track dihapus saat hilang
        match_thresh=0.8,                 # IoU threshold untuk matching
        proximity_thresh=0.5,
        appearance_thresh=0.25,
        with_reid=True,                   # aktifkan Re-ID appearance matching
    )

    tracked_detections = []
    human_detections   = []

    print('Running YOLOv8 + BoT-SORT tracking...')
    prog_bar = mmengine.ProgressBar(len(frame_paths))

    for frame_path in frame_paths:
        frame_bgr = cv2.imread(frame_path)  # BoT-SORT butuh frame BGR asli untuk Re-ID

        # Step 1: YOLOv8 deteksi
        yolo_res = detector(frame_bgr, classes=[0], verbose=False)
        boxes    = yolo_res[0].boxes.xyxy.cpu().numpy()   # [N, 4]
        scores   = yolo_res[0].boxes.conf.cpu().numpy()   # [N]

        # Format BoT-SORT: [x1, y1, x2, y2, score, class]
        if len(boxes) > 0:
            dets = np.hstack([boxes, scores[:, None], np.zeros((len(boxes), 1))])
        else:
            dets = np.empty((0, 6))

        # Step 2: BoT-SORT update
        # Output tracks: [x1, y1, x2, y2, track_id, score, class, ...]
        tracks = tracker.update(dets, frame_bgr)

        # Step 3: Kemas output
        frame_tracks    = []
        frame_human_det = []

        if len(tracks) > 0:
            for t in tracks:
                x1, y1, x2, y2 = t[0], t[1], t[2], t[3]
                tid   = int(t[4])
                score = float(t[5])
                frame_tracks.append({'track_id': tid, 'bbox': [x1, y1, x2, y2], 'score': score})
                frame_human_det.append([x1, y1, x2, y2, score])

        tracked_detections.append(frame_tracks)
        det_arr = np.array(frame_human_det) if frame_human_det else np.zeros((0, 5))
        human_detections.append(det_arr)

        prog_bar.update()

    print(f'\nSelesai. Total frame: {len(frame_paths)}')
    return tracked_detections, human_detections

## 4. Clip Extraction berbasis Track ID

### Perbedaan inti dengan versi lama (IoU matching):

| Aspek | Versi Lama | Versi BoT-SORT |
|---|---|---|
| Identifikasi orang | Perbandingan IoU bbox | `track_id` langsung dari BoT-SORT |
| Saat dua orang crossing | ❌ Bisa tertukar | ✅ ID tetap konsisten |
| Occlusion singkat | ❌ Keypoint hilang/salah | ✅ Kalman prediksi posisi |
| Nama clip di PKL | `videoname_timestamp_i` | `videoname_t{ts}_id{track_id}` |

In [8]:
def build_track_pose_index(tracked_detections, pose_results):
    """
    Bangun index: per frame, track_id → index pose di pose_results.
    IoU digunakan hanya untuk menemukan index pose yang sesuai dengan track bbox,
    bukan untuk menentukan identitas orang (itu sudah dijamin track_id).
    """
    track_pose_index = []
    for frame_idx, (frame_tracks, frame_poses) in enumerate(
            zip(tracked_detections, pose_results)):
        id_to_pose = {}
        if not frame_tracks or len(frame_poses.get('keypoints', [])) == 0:
            track_pose_index.append(id_to_pose)
            continue
        pose_bboxes = frame_poses['bboxes']
        for track in frame_tracks:
            tid = track['track_id']
            best_iou, best_pidx = -1, -1
            for pidx, pbbox in enumerate(pose_bboxes):
                iou = _cal_iou(track['bbox'], pbbox)
                if iou > best_iou:
                    best_iou, best_pidx = iou, pidx
            if best_pidx >= 0 and best_iou > 0.1:
                id_to_pose[tid] = best_pidx
        track_pose_index.append(id_to_pose)
    return track_pose_index


def skeleton_based_stdet_botsort(predict_stepsize, video,
                                  tracked_detections, pose_results,
                                  num_frame, clip_len, frame_interval, h, w):
    window_size = clip_len * frame_interval
    assert clip_len % 2 == 0, 'clip_len harus genap'

    timestamps = np.arange(
        window_size // 2,
        num_frame + 1 - window_size // 2,
        predict_stepsize
    )

    print('Building track→pose index...')
    track_pose_index = build_track_pose_index(tracked_detections, pose_results)

    skeleton_predictions = []
    skeleton_datasets    = []

    print('Extracting clips per track_id...')
    prog_bar = mmengine.ProgressBar(len(timestamps))

    for timestamp in timestamps:
        start_frame   = timestamp - (clip_len // 2 - 1) * frame_interval
        frame_inds    = list(start_frame + np.arange(0, window_size, frame_interval) - 1)
        frame_inds    = [max(0, min(int(fi), num_frame - 1)) for fi in frame_inds]
        n_clip_frames = len(frame_inds)

        center_idx    = int(timestamp) - 1
        active_tracks = tracked_detections[center_idx]

        if not active_tracks:
            skeleton_predictions.append(None)
            prog_bar.update()
            continue

        skeleton_prediction = []

        for i, track in enumerate(active_tracks):
            tid = track['track_id']
            skeleton_prediction.append([])

            keypoint       = np.zeros((1, n_clip_frames, 17, 2))
            keypoint_score = np.zeros((1, n_clip_frames, 17))

            for j, fi in enumerate(frame_inds):
                pose_idx = track_pose_index[fi].get(tid, None)
                if pose_idx is not None:
                    keypoint[0, j]       = pose_results[fi]['keypoints'][pose_idx]
                    keypoint_score[0, j] = pose_results[fi]['keypoint_scores'][pose_idx]

            # ── Format frame_dir & CSV ID ────────────────────────────────
            # frame_dir: namafile_t{timestamp}_id{track_id}  (untuk matching di cell labeling)
            # CSV ID   : timestamp + (i+1)*0.001             (untuk tampil di video, sama seperti kode asli)
            # Contoh   : timestamp=46, orang ke-2 → CSV="annotate!,46.002", frame_dir="video_t46_id5"
            frame_dir_name = (
                osp.splitext(osp.basename(video))[0]
                + f'_t{int(timestamp)}_id{tid}'
            )
            csv_id = int(timestamp) + (i + 1) * 0.001  # ← format asli: 46.001, 46.002, dst

            fake_anno = dict(
                frame_dir=frame_dir_name,
                label=-1,
                img_shape=(h, w),
                original_shape=(h, w),
                num_clips=1,
                total_frames=n_clip_frames,
                keypoint=keypoint,
                keypoint_score=keypoint_score,
                track_id=tid,           # disimpan untuk referensi
                csv_id=csv_id,          # disimpan untuk matching saat labeling
            )

            skeleton_datasets.append(fake_anno)
            skeleton_prediction[i].append(('annotate!', csv_id))  # format: annotate!,46.002

        skeleton_predictions.append(skeleton_prediction)
        prog_bar.update()

    return timestamps, skeleton_predictions, skeleton_datasets


## 5. Visualisasi (dengan Track ID overlay)

In [9]:
def visualize(pose_config_path, frames, annotations, pose_data_samples,
              action_result, plate=PLATEBLUE, max_num=5,
              tracked_detections=None, output_timestamps=None,
              all_timestamps=None, predict_stepsize=4):
    """
    Visualisasi video normal (semua frame), overlay keterangan hanya di anchor frame.

    - Semua frame: skeleton pose + bbox BoT-SORT mengikuti objek
    - Hanya anchor frame (kelipatan predict_stepsize): tampil label CSV ID + nomor frame
    """
    assert max_num + 1 <= len(plate)
    frames_ = cp.deepcopy(frames)
    frames_ = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    h, w, _ = frames[0].shape
    bahaya  = ['melempar', 'membidik senapan', 'membidik pistol',
               'memukul', 'menendang', 'menusuk']

    # Set frame index yang merupakan anchor (tiap predict_stepsize)
    anchor_set = set(all_timestamps.tolist()) if all_timestamps is not None else set()

    # ── 1. Skeleton pose untuk semua frame ──────────────────────────────
    if pose_data_samples and any(d is not None for d in pose_data_samples):
        pose_cfg   = mmengine.Config.fromfile(pose_config_path)
        visualizer = VISUALIZERS.build(
            pose_cfg.visualizer | {'line_width': 1, 'bbox_color': (101, 193, 255), 'radius': 2})
        visualizer.set_dataset_meta(
            next(d for d in pose_data_samples if d is not None).dataset_meta)
        for i, (d, f) in enumerate(zip(pose_data_samples, frames_)):
            if d is None:
                continue
            visualizer.add_datasample('result', f, data_sample=d,
                                       draw_gt=False, draw_heatmap=False,
                                       draw_bbox=False,
                                       draw_pred=True, show=False, wait_time=0,
                                       out_file=None, kpt_thr=0.3)
            frames_[i] = visualizer.get_image()

    # ── 2. Overlay per frame ─────────────────────────────────────────────
    for i, frame in enumerate(frames_):
        actual_frame_num = int(output_timestamps[i]) if output_timestamps is not None else i + 1
        fi = min(actual_frame_num - 1, len(tracked_detections) - 1)
        is_anchor = actual_frame_num in anchor_set

        # Bbox BoT-SORT mengikuti objek di SEMUA frame
        if tracked_detections and fi >= 0:
            for t_idx, trk in enumerate(tracked_detections[fi]):
                x1, y1, x2, y2 = [int(v) for v in trk['bbox']]
                tid = trk['track_id']

                # Bbox cyan tipis di semua frame
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 200), 2)

                # Track ID di atas bbox — semua frame
                cv2.putText(frame, f'ID:{tid}',
                            (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 200), 1)

                # ── Hanya di anchor frame: tampilkan CSV ID & label ──────
                if is_anchor and annotations and i < len(annotations):
                    ann_frame = annotations[i]
                    if ann_frame and t_idx < len(ann_frame):
                        ann   = ann_frame[t_idx]
                        label = ann[1]
                        score = ann[2]
                        for kl, lb in enumerate(label):
                            if kl >= max_num:
                                break
                            text     = abbrev(lb)
                            location = (x1, y1 + 20 + kl * 20)
                            tw, th   = cv2.getTextSize(text, FONTFACE, 0.6, 1)[0]
                            cv2.rectangle(frame,
                                          (location[0], location[1] - th - 2),
                                          (location[0] + tw, location[1] + 2),
                                          plate[kl + 1], -1)
                            fc = (255, 0, 0) if lb in bahaya else (255, 255, 255)
                            cv2.putText(frame, text, location,
                                        FONTFACE, 0.6, fc, 1, LINETYPE)

        # ── Nomor frame — hanya di anchor frame ─────────────────────────
        if is_anchor:
            frame_text  = f'Frame: {actual_frame_num}'
            (tw, th), _ = cv2.getTextSize(frame_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            cv2.rectangle(frame, (8, 8), (18 + tw, 18 + th + 8), (0, 0, 0), -1)
            cv2.putText(frame, frame_text, (12, 12 + th),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 1, cv2.LINE_AA)

    return frames_


## 6. Main Pipeline (Batch)

Semua video di folder `data_in` diproses otomatis.
Hasil per video: `.csv` + `.pkl` + video output untuk anotasi.

In [26]:
# Scan semua video .mp4 di INPUT_DIR (rekursif)
video_list = sorted(glob.glob(os.path.join(INPUT_DIR, '**/*.avi'), recursive=True)+
                    glob.glob(os.path.join(INPUT_DIR, '**/*.mp4'), recursive=True))

print(f'Total video ditemukan: {len(video_list)}')
print()
for i, v in enumerate(video_list):
    base     = os.path.splitext(os.path.basename(v))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{base}_out.mp4')
    status   = '✓ sudah diproses' if os.path.exists(out_path) else '○ belum diproses'
    print(f'  [{i+1:02d}] {status} — {os.path.basename(v)}')


Total video ditemukan: 21

  [01] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian005.mp4
  [02] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian009.mp4
  [03] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian011.mp4
  [04] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian012.mp4
  [05] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian013.mp4
  [06] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian019.mp4
  [07] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian020.mp4
  [08] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian021.mp4
  [09] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian031.mp4
  [10] ○ belum diproses — Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian032.mp4
  [11] ○ belum diproses — Top 60 

In [27]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PKL_DIR,   exist_ok=True)

total    = len(video_list)
success  = 0
skipped  = 0
failed   = 0

for video_idx, video in enumerate(video_list):
    base_name    = os.path.splitext(os.path.basename(video))[0]
    ann_filename = os.path.join(PKL_DIR,    f'{base_name}.csv')
    pkl_filename = os.path.join(PKL_DIR,    f'{base_name}.pkl')
    out_filename = os.path.join(OUTPUT_DIR, f'{base_name}_out.mp4')

    print(f'\n{"="*55}')
    print(f'[{video_idx+1}/{total}] {base_name}')
    print(f'{"="*55}')

    # Skip jika sudah diproses
    if os.path.exists(out_filename):
        print(f'  ⏭  Skip — sudah diproses sebelumnya.')
        skipped += 1
        continue

    try:
        # ── STEP 1: Ekstrak Frame ──────────────────────────────────
        print('  [1/6] Ekstrak frame...')
        tmp_dir = tempfile.TemporaryDirectory()
        frame_paths, original_frames = frame_extract(
            video, 480, out_dir=tmp_dir.name)
        num_frame = len(frame_paths)
        h, w, _   = original_frames[0].shape
        del original_frames
        gc.collect()
        print(f'        {num_frame} frame | {w}x{h}')

        # ── STEP 2: YOLOv8 + BoT-SORT ─────────────────────────────
        print('  [2/6] YOLOv8 + BoT-SORT tracking...')
        tracked_detections, human_detections = run_detection_and_tracking(
            frame_paths, detector, det_score_thr,
            reid_weights=BOTSORT_REID_WEIGHTS,
            track_device=BOTSORT_DEVICE,
            half=BOTSORT_HALF
        )
        avg_p = sum(len(f) for f in tracked_detections) / max(len(tracked_detections), 1)
        print(f'        Rata-rata {avg_p:.1f} orang per frame')

        # ── STEP 3: HRNet Pose Estimation ─────────────────────────
        print('  [3/6] HRNet pose estimation...')
        pose_results, pose_datasample = pose_inference(
            pose_config, pose_checkpoint,
            frame_paths, human_detections, device=device)
        torch.cuda.empty_cache()

        # ── STEP 4: Clip Extraction ────────────────────────────────
        print('  [4/6] Clip extraction...')
        timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet_botsort(
            predict_stepsize, video,
            tracked_detections, pose_results,
            num_frame, predict_stepsize, 1, h, w
        )
        print(f'        {len(skeleton_datasets)} clip dihasilkan')

        # ── STEP 5: Simpan CSV + PKL ───────────────────────────────
        print('  [5/6] Simpan CSV + PKL...')
        anno = ''
        for clip in stdet_preds:
            if clip is None:
                continue
            for person_attr in clip:
                anno += f'{person_attr[0][0]},{person_attr[0][1]:.3f}\n'

        with open(ann_filename, 'w') as f:
            f.write(anno)
        mmengine.dump(skeleton_datasets, pkl_filename)
        print(f'        CSV : {ann_filename}')
        print(f'        PKL : {pkl_filename}')

        # ── STEP 6: Visualisasi ────────────────────────────────────
        print('  [6/6] Render video anotasi...')
        anchor_anno_map = {}
        for timestamp, prediction in zip(timestamps, stdet_preds):
            if prediction is None:
                continue
            frame_anno = []
            for person_pred in prediction:
                if not person_pred:
                    continue
                label_str = person_pred[0][0]
                csv_id    = person_pred[0][1]
                frame_anno.append((
                    np.array([0, 0, 1, 1], dtype=np.float32),
                    [f'{label_str}: {csv_id:.3f}'],
                    [1.0]
                ))
            anchor_anno_map[int(timestamp)] = frame_anno

        output_timestamps = np.arange(1, num_frame + 1, dtype=np.int64)
        annotations_all   = [anchor_anno_map.get(int(ts), None) for ts in output_timestamps]
        frames_all        = [cv2.imread(frame_paths[min(ts-1, len(frame_paths)-1)])
                             for ts in output_timestamps]
        pose_ds_all       = [pose_datasample[min(ts-1, len(pose_datasample)-1)]
                             for ts in output_timestamps]

        vis_frames = visualize(
            pose_config, frames_all, annotations_all, pose_ds_all, None,
            tracked_detections=tracked_detections,
            output_timestamps=output_timestamps,
            all_timestamps=timestamps,
            predict_stepsize=predict_stepsize
        )

        vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
        vid.write_videofile(out_filename, logger=None)
        print(f'        Video: {out_filename}')

        # Cleanup per video
        tmp_dir.cleanup()
        del frames_all, vis_frames, pose_results, pose_datasample
        del tracked_detections, human_detections, skeleton_datasets
        del timestamps, stdet_preds, anchor_anno_map, annotations_all
        gc.collect()
        torch.cuda.empty_cache()

        success += 1
        print(f'  ✓ Selesai!')

    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        try:
            tmp_dir.cleanup()
        except:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        failed += 1
        continue

# Ringkasan akhir
print(f'\n{"="*55}')
print(f'SELESAI — Ringkasan:')
print(f'  Berhasil : {success} video')
print(f'  Di-skip  : {skipped} video (sudah diproses)')
print(f'  Gagal    : {failed} video')
print(f'{"="*55}')
print(f'Video output → {OUTPUT_DIR}/')
print(f'Selanjutnya  → buka tiap .csv, isi label aksi, lalu jalankan cell labeling.')



[1/21] Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian005
  [1/6] Ekstrak frame...


2026-05-26 05:08:22.077 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.77 🚀 Python-3.8.18 torch-2.2.1
CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
2026-05-26 05:08:22.118 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from 01_asset_tools/models/osnet_x0_25_msmt17.pt


        299 frame | 853x480
  [2/6] YOLOv8 + BoT-SORT tracking...
Running YOLOv8 + BoT-SORT tracking...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 24.6 task/s, elapsed: 12s, ETA:     0s
Selesai. Total frame: 299
        Rata-rata 4.0 orang per frame
  [3/6] HRNet pose estimation...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 11.1 task/s, elapsed: 27s, ETA:     0s
  [4/6] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 74/74, 2343.2 task/s, elapsed: 0s, ETA:     0s        292 clip dihasilkan
  [5/6] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian005.csv
        PKL : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian005.pkl
  [6/6] Ren

2026-05-26 05:09:07.720 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.77 🚀 Python-3.8.18 torch-2.2.1
CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
2026-05-26 05:09:07.762 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from 01_asset_tools/models/osnet_x0_25_msmt17.pt


        299 frame | 853x480
  [2/6] YOLOv8 + BoT-SORT tracking...
Running YOLOv8 + BoT-SORT tracking...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 25.0 task/s, elapsed: 12s, ETA:     0s
Selesai. Total frame: 299
        Rata-rata 4.1 orang per frame
  [3/6] HRNet pose estimation...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 11.4 task/s, elapsed: 26s, ETA:     0s
  [4/6] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 74/74, 1547.9 task/s, elapsed: 0s, ETA:     0s        308 clip dihasilkan
  [5/6] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian009.csv
        PKL : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian009.pkl
  [6/6] Ren

...

        299 frame | 853x480
  [2/6] YOLOv8 + BoT-SORT tracking...
Running YOLOv8 + BoT-SORT tracking...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 23.1 task/s, elapsed: 13s, ETA:     0s
Selesai. Total frame: 299
        Rata-rata 4.0 orang per frame
  [3/6] HRNet pose estimation...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 10.7 task/s, elapsed: 28s, ETA:     0s
  [4/6] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 74/74, 1601.9 task/s, elapsed: 0s, ETA:     0s        290 clip dihasilkan
  [5/6] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian078.csv
        PKL : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian078.pkl
  [6/6] Ren

2026-05-26 05:23:24.251 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.77 🚀 Python-3.8.18 torch-2.2.1
CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
2026-05-26 05:23:24.291 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from 01_asset_tools/models/osnet_x0_25_msmt17.pt


        299 frame | 853x480
  [2/6] YOLOv8 + BoT-SORT tracking...
Running YOLOv8 + BoT-SORT tracking...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 22.3 task/s, elapsed: 13s, ETA:     0s
Selesai. Total frame: 299
        Rata-rata 5.6 orang per frame
  [3/6] HRNet pose estimation...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 9.8 task/s, elapsed: 31s, ETA:     0s
  [4/6] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 74/74, 1457.4 task/s, elapsed: 0s, ETA:     0s        416 clip dihasilkan
  [5/6] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian079.csv
        PKL : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian079.pkl
  [6/6] Ren

2026-05-26 05:24:15.122 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.77 🚀 Python-3.8.18 torch-2.2.1
CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
2026-05-26 05:24:15.163 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from 01_asset_tools/models/osnet_x0_25_msmt17.pt


        299 frame | 853x480
  [2/6] YOLOv8 + BoT-SORT tracking...
Running YOLOv8 + BoT-SORT tracking...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 22.4 task/s, elapsed: 13s, ETA:     0s
Selesai. Total frame: 299
        Rata-rata 4.5 orang per frame
  [3/6] HRNet pose estimation...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 299/299, 10.3 task/s, elapsed: 29s, ETA:     0s
  [4/6] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 74/74, 1513.3 task/s, elapsed: 0s, ETA:     0s        326 clip dihasilkan
  [5/6] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian080.csv
        PKL : 03_dataset/data_in/1_csv/Top 60 Crazy Knockouts 2023 in Kickboxing & Muay Thai_bagian080.pkl
  [6/6] Ren

## 7. Add Label ke Dataset
Edit file CSV dulu (ganti `annotate!` → nama label), lalu jalankan cell berikut.

In [ ]:
import csv

def load_label_map(file_path):
    lines = open(file_path).readlines()
    return {x[1]: int(x[0]) for x in [l.strip().split(': ') for l in lines]}

stdet_label_map = load_label_map(label_map_stdet)
stdet_label_map

In [ ]:
custom_annos = []
with open(ann_filename, newline='') as csvfile:
    for row in csv.reader(csvfile, delimiter=','):
        if row[0] in ('none', 'annotate!'):
            continue
        custom_annos.append([float(row[1]), stdet_label_map[row[0]]])

print(f'Clip teranotasi: {len(custom_annos)}')

In [ ]:
skeleton_datasets = mmengine.load(pkl_filename)
custom_dataset    = []

for idx, ann in enumerate(custom_annos):
    csv_id_target = ann[0]   # float: misal 46.002

    for data in skeleton_datasets:
        # Cocokkan berdasarkan csv_id yang disimpan di pkl
        if abs(data.get('csv_id', -1) - csv_id_target) < 1e-6:
            labeled = cp.deepcopy(data)
            labeled['frame_dir'] += f'_{idx}'
            labeled['label']      = int(ann[1])
            labeled['clip_len']   = data['total_frames']
            custom_dataset.append(labeled)
            break

print(f'Dataset berlabel: {len(custom_dataset)} clip')


In [ ]:
# Simpan PKL berlabel ke folder to-combine
# pkl_final = 'data/skeleton/to-combine/data_try1.pkl'
# mmengine.dump(custom_dataset, pkl_final)
# print(f'Disimpan: {pkl_final}')

## 8. Combine PKL + Split Train/Val

In [ ]:
pickles_path = 'data/skeleton/to-combine'
split_ratio  = 0.5
combined_pkl = 'data/skeleton/ciis_' + str(split_ratio).replace('.', 's') + '_v1.pkl'

custom_datasets = dict(
    split=dict(xsub_train=[], xsub_val=[], xview_train=[], xview_val=[]),
    annotations=[]
)

for file in os.listdir(pickles_path):
    if not file.endswith('.pkl'):
        continue
    print(file)
    ds = mmengine.load(os.path.join(pickles_path, file))
    for i, data in enumerate(ds):
        custom_datasets['annotations'].append(data)
        split_key = 'train' if (i % 10) < (split_ratio * 10) else 'val'
        custom_datasets['split'][f'xsub_{split_key}'].append(data['frame_dir'])
        custom_datasets['split'][f'xview_{split_key}'].append(data['frame_dir'])

mmengine.dump(custom_datasets, combined_pkl)
n_tr = len(custom_datasets['split']['xsub_train'])
n_va = len(custom_datasets['split']['xsub_val'])
print(f'\nCombined PKL: {combined_pkl}')
print(f'Train: {n_tr} | Val: {n_va} | Total: {n_tr + n_va}')